# Laboratorio 8: Avance de SVM en Python

Este notebook cubre **únicamente los puntos 1 al 7 del avance/pasaporte** del laboratorio.
No incluye todavía la comparación final con modelos anteriores ni la parte de SVR.

**Objetivo del avance**
- Cargar y explorar `listings.RData`.
- Limpiar la variable de precio.
- Crear o reutilizar la variable categórica `price_cat`.
- Mantener una división reproducible de entrenamiento y prueba.
- Entrenar varios modelos SVM de clasificación.
- Evaluar métricas, matrices de confusión y sobreajuste/desajuste.


In [ ]:
from pathlib import Path
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import pyreadr
import seaborn as sns
import matplotlib

RUN_MODE = os.getenv("LAB8_SVM_RUN_MODE", "notebook")
if RUN_MODE == "script":
    matplotlib.use("Agg")

    def display(*args, **kwargs):
        return None

    def Markdown(text):
        return text
else:
    from IPython.display import Markdown, display

import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 3074
DATA_PATH = Path("listings.RData")
OUTPUT_DIR = Path("salidas_lab8_svm_python")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Archivo de datos: {DATA_PATH.resolve()}")


## 1. Carga del dataset y revisión inicial

Primero se carga el objeto `listings` desde el archivo `RData`. Después se revisa el tamaño,
la estructura general, los tipos de variables, el porcentaje de faltantes y la forma en que
viene almacenado el precio.


In [ ]:
rdata = pyreadr.read_r(DATA_PATH)
print("Objetos encontrados en el RData:", list(rdata.keys()))

df_raw = rdata["listings"].copy()

structure_summary = pd.DataFrame(
    {
        "total_filas": [df_raw.shape[0]],
        "total_columnas": [df_raw.shape[1]],
        "columnas_numericas": [df_raw.select_dtypes(include=[np.number]).shape[1]],
        "columnas_objeto": [df_raw.select_dtypes(include=["object"]).shape[1]],
        "columnas_datetime": [df_raw.select_dtypes(include=["datetime"]).shape[1]],
    }
)

missing_summary = (
    df_raw.isna()
    .sum()
    .rename("faltantes")
    .to_frame()
    .assign(
        porcentaje_faltante=lambda x: (x["faltantes"] / len(df_raw) * 100).round(2)
    )
    .sort_values("faltantes", ascending=False)
)

display(structure_summary)
display(df_raw.head(3))
display(missing_summary.head(15))

structure_summary.to_csv(OUTPUT_DIR / "dataset_structure.csv", index=False)
missing_summary.reset_index(names="variable").to_csv(
    OUTPUT_DIR / "missing_summary.csv", index=False
)

print("Columnas que contienen la palabra 'price':")
print([col for col in df_raw.columns if "price" in col.lower()])
print("\nPrimeros valores originales de 'price':")
print(df_raw["price"].head(10).tolist())


## 2. Transformaciones necesarias para usar SVM

Para aplicar SVM de clasificación correctamente se requieren varias transformaciones:

- **Limpieza de `price`**: el precio viene como texto con símbolos como `$` y comas, por lo que debe convertirse a número.
- **Tratamiento de valores faltantes**: SVM no acepta `NaN` directamente, así que se imputan variables numéricas y categóricas.
- **Codificación de variables categóricas**: los kernels de SVM trabajan con variables numéricas, así que las categóricas deben pasarse a one-hot encoding.
- **Escalamiento de variables numéricas**: SVM es sensible a la escala porque la separación depende de distancias y productos punto. Si una variable tiene magnitudes mucho mayores, puede dominar la frontera de decisión y sesgar el margen.
- **Reducción de variables no útiles**: se excluyen identificadores, URLs y texto libre porque no son adecuados para este avance y aumentarían demasiado la dimensionalidad.

En este notebook, la imputación, la codificación y el escalamiento se ajustan **solo con el conjunto de entrenamiento** mediante `Pipeline` y `ColumnTransformer`, para evitar **data leakage**.


In [ ]:
def clean_price(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype("string").str.replace(r"[$,]", "", regex=True),
        errors="coerce",
    )


def parse_percent(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )
    return pd.to_numeric(cleaned.str.replace("%", "", regex=False), errors="coerce") / 100.0


def parse_yes_no(series: pd.Series) -> pd.Series:
    mapping = {"t": "si", "f": "no", "true": "si", "false": "no", "1": "si", "0": "no"}
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA, "n/a": pd.NA, "na": pd.NA, "null": pd.NA})
        .map(mapping)
        .astype("string")
    )


def parse_bathrooms_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"", "n/a", "na", "null"}:
        return np.nan
    if "half" in text:
        return 0.5
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else np.nan


def bathroom_privacy(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if "shared" in text:
        return "compartido"
    if "private" in text:
        return "privado"
    return "no_especificado"


def count_amenities(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    if text in {"", "[]"}:
        return 0
    return text.count('", "') + 1


def build_analysis_dataframe(df: pd.DataFrame):
    data = df.copy()
    data["listing_id"] = data["id"]
    data["price_num"] = clean_price(data["price"])
    data = data.loc[data["price_num"].notna() & (data["price_num"] > 0)].copy()

    q1 = data["price_num"].quantile(1 / 3)
    q2 = data["price_num"].quantile(2 / 3)
    data["price_cat"] = pd.cut(
        data["price_num"],
        bins=[-np.inf, q1, q2, np.inf],
        labels=["barata", "media", "cara"],
        include_lowest=True,
    )

    data["host_response_time"] = (
        data["host_response_time"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )
    data["host_response_rate"] = parse_percent(data["host_response_rate"])
    data["host_acceptance_rate"] = parse_percent(data["host_acceptance_rate"])
    data["host_is_superhost"] = parse_yes_no(data["host_is_superhost"])
    data["host_identity_verified"] = parse_yes_no(data["host_identity_verified"])
    data["instant_bookable"] = parse_yes_no(data["instant_bookable"])

    data["neighbourhood_group_cleansed"] = (
        data["neighbourhood_group_cleansed"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )
    data["neighbourhood_cleansed"] = (
        data["neighbourhood_cleansed"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )
    data["property_type"] = (
        data["property_type"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )
    data["room_type"] = (
        data["room_type"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NULL": pd.NA})
    )

    data["bathrooms_num"] = data["bathrooms"].combine_first(
        data["bathrooms_text"].map(parse_bathrooms_text)
    )
    data["bathroom_privacy"] = data["bathrooms_text"].map(bathroom_privacy).astype("string")
    data["amenities_count"] = data["amenities"].map(count_amenities)

    data["minimum_nights_log"] = np.log1p(data["minimum_nights"].clip(upper=365))
    data["maximum_nights_log"] = np.log1p(data["maximum_nights"].clip(upper=3650))
    data["number_of_reviews_log"] = np.log1p(data["number_of_reviews"])
    data["reviews_per_month"] = pd.to_numeric(data["reviews_per_month"], errors="coerce")

    review_cols = [
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value",
    ]
    for col in review_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    count_cols = [
        "calculated_host_listings_count",
        "calculated_host_listings_count_entire_homes",
        "calculated_host_listings_count_private_rooms",
        "calculated_host_listings_count_shared_rooms",
    ]
    for col in count_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    data["calculated_host_listings_count_log"] = np.log1p(
        data["calculated_host_listings_count"]
    )

    for date_col in ["host_since", "first_review", "last_review", "last_scraped"]:
        data[date_col] = pd.to_datetime(data[date_col], errors="coerce")

    data["host_tenure_days"] = (
        data["last_scraped"] - data["host_since"]
    ).dt.days
    data["days_since_first_review"] = (
        data["last_scraped"] - data["first_review"]
    ).dt.days
    data["days_since_last_review"] = (
        data["last_scraped"] - data["last_review"]
    ).dt.days

    numeric_features = [
        "host_response_rate",
        "host_acceptance_rate",
        "accommodates",
        "bathrooms_num",
        "bedrooms",
        "beds",
        "amenities_count",
        "minimum_nights_log",
        "maximum_nights_log",
        "availability_30",
        "availability_60",
        "availability_90",
        "availability_365",
        "number_of_reviews_log",
        "reviews_per_month",
        "review_scores_rating",
        "review_scores_accuracy",
        "review_scores_cleanliness",
        "review_scores_checkin",
        "review_scores_communication",
        "review_scores_location",
        "review_scores_value",
        "calculated_host_listings_count_log",
        "calculated_host_listings_count_entire_homes",
        "calculated_host_listings_count_private_rooms",
        "calculated_host_listings_count_shared_rooms",
        "host_tenure_days",
        "days_since_first_review",
        "days_since_last_review",
        "latitude",
        "longitude",
    ]

    categorical_features = [
        "host_response_time",
        "host_is_superhost",
        "host_identity_verified",
        "neighbourhood_group_cleansed",
        "neighbourhood_cleansed",
        "property_type",
        "room_type",
        "bathroom_privacy",
        "instant_bookable",
    ]

    selected_columns = ["listing_id", "price_num", "price_cat"] + numeric_features + categorical_features
    data = data[selected_columns].copy()

    for col in numeric_features:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    for col in categorical_features:
        data[col] = data[col].astype("object")
        data.loc[pd.isna(data[col]), col] = np.nan

    metadata = {
        "price_min": float(data["price_num"].min()),
        "price_q1": float(q1),
        "price_q2": float(q2),
        "price_max": float(data["price_num"].max()),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    }
    return data, metadata


analysis_df, metadata = build_analysis_dataframe(df_raw)

print("Filas con precio válido:", len(analysis_df))
print(
    "Cortes de price_cat:",
    {
        "min": round(metadata["price_min"], 2),
        "q1": round(metadata["price_q1"], 2),
        "q2": round(metadata["price_q2"], 2),
        "max": round(metadata["price_max"], 2),
    },
)
display(analysis_df.head(3))


In [ ]:
class_distribution = (
    analysis_df["price_cat"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("price_cat")
    .reset_index(name="frecuencia")
)
class_distribution["proporcion"] = (
    class_distribution["frecuencia"] / class_distribution["frecuencia"].sum()
).round(4)

feature_missing = (
    analysis_df.isna()
    .sum()
    .rename("faltantes")
    .to_frame()
    .assign(
        porcentaje_faltante=lambda x: (x["faltantes"] / len(analysis_df) * 100).round(2)
    )
    .sort_values("faltantes", ascending=False)
)

display(class_distribution)
display(feature_missing.head(15))

class_distribution.to_csv(OUTPUT_DIR / "price_category_distribution_full.csv", index=False)
feature_missing.reset_index(names="variable").to_csv(
    OUTPUT_DIR / "feature_missing_summary.csv", index=False
)


## 3. Conjunto de entrenamiento y prueba

En la carpeta actual no se encontró un notebook o script previo con una división guardada de `train/test`.
Por esa razón se crea una partición reproducible con `random_state=3074`.

Como el dataset tiene decenas de miles de registros con precio válido y los SVM no lineales son costosos
computacionalmente, se toma primero una **muestra estratificada reproducible** para clasificación.
Esa misma muestra y esa misma división deben mantenerse en las siguientes etapas si se desea comparar modelos
en igualdad de condiciones.


In [ ]:
split_metadata_path = OUTPUT_DIR / "train_test_split_metadata.json"
max_sample_size = 6000

if split_metadata_path.exists():
    split_metadata = json.loads(split_metadata_path.read_text(encoding="utf-8"))
    sample_ids = set(split_metadata["sample_ids"])
    train_ids = set(split_metadata["train_ids"])
    test_ids = set(split_metadata["test_ids"])

    sampled_df = analysis_df[analysis_df["listing_id"].isin(sample_ids)].copy()
    train_df = sampled_df[sampled_df["listing_id"].isin(train_ids)].copy()
    test_df = sampled_df[sampled_df["listing_id"].isin(test_ids)].copy()
    split_message = "Se reutilizó la muestra y la división train/test guardada previamente."
else:
    if len(analysis_df) > max_sample_size:
        sampled_df, _ = train_test_split(
            analysis_df,
            train_size=max_sample_size,
            stratify=analysis_df["price_cat"],
            random_state=RANDOM_STATE,
        )
        sampled_df = sampled_df.copy()
        sampled = True
    else:
        sampled_df = analysis_df.copy()
        sampled = False

    train_df, test_df = train_test_split(
        sampled_df,
        test_size=0.2,
        stratify=sampled_df["price_cat"],
        random_state=RANDOM_STATE,
    )

    split_metadata = {
        "sampled": sampled,
        "sample_size": int(len(sampled_df)),
        "train_size": int(len(train_df)),
        "test_size": int(len(test_df)),
        "sample_ids": sampled_df["listing_id"].tolist(),
        "train_ids": train_df["listing_id"].tolist(),
        "test_ids": test_df["listing_id"].tolist(),
        "random_state": RANDOM_STATE,
    }
    split_metadata_path.write_text(
        json.dumps(split_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    split_message = "Se creó una nueva muestra estratificada y una nueva división train/test reproducible."

train_df = train_df.sort_values("listing_id").reset_index(drop=True)
test_df = test_df.sort_values("listing_id").reset_index(drop=True)

train_distribution = train_df["price_cat"].value_counts().sort_index().rename("frecuencia")
test_distribution = test_df["price_cat"].value_counts().sort_index().rename("frecuencia")

print(split_message)
print(f"Tamaño de la muestra de modelado: {len(train_df) + len(test_df)}")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
display(train_distribution.to_frame("train"))
display(test_distribution.to_frame("test"))

train_distribution.rename_axis("price_cat").reset_index().to_csv(
    OUTPUT_DIR / "train_class_distribution.csv", index=False
)
test_distribution.rename_axis("price_cat").reset_index().to_csv(
    OUTPUT_DIR / "test_class_distribution.csv", index=False
)


## 4. Pipeline de preprocesamiento

El pipeline separa variables numéricas y categóricas:

- **Numéricas**: imputación por mediana + escalamiento con `StandardScaler`.
- **Categóricas**: imputación por moda + `OneHotEncoder`.

Para controlar categorías raras en variables como `neighbourhood_cleansed` y `property_type`,
el `OneHotEncoder` agrupa categorías infrecuentes usando `min_frequency=0.01`.
Esto ayuda a reducir dimensionalidad sin usar información del conjunto de prueba.


In [ ]:
numeric_features = metadata["numeric_features"]
categorical_features = metadata["categorical_features"]
feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()
y_train = train_df["price_cat"].copy()
y_test = test_df["price_cat"].copy()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=0.01,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

model_configs = [
    {"modelo": "svm_linear_c0_5", "kernel": "linear", "C": 0.5},
    {"modelo": "svm_linear_c2", "kernel": "linear", "C": 2.0},
    {"modelo": "svm_rbf_c1_g001", "kernel": "rbf", "C": 1.0, "gamma": 0.01},
    {"modelo": "svm_rbf_c3_g002", "kernel": "rbf", "C": 3.0, "gamma": 0.02},
    {"modelo": "svm_poly_c1_d2", "kernel": "poly", "C": 1.0, "gamma": 0.01, "degree": 2, "coef0": 1.0},
    {"modelo": "svm_poly_c2_d3", "kernel": "poly", "C": 2.0, "gamma": 0.01, "degree": 3, "coef0": 1.0},
]

display(pd.DataFrame(model_configs))


In [ ]:
labels = ["barata", "media", "cara"]


def compute_metric_block(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


def confusion_summary(cm, class_labels):
    recalls = np.divide(
        np.diag(cm),
        cm.sum(axis=1),
        out=np.zeros(len(class_labels), dtype=float),
        where=cm.sum(axis=1) != 0,
    )
    best_class = class_labels[int(np.argmax(recalls))]
    worst_class = class_labels[int(np.argmin(recalls))]

    temp = cm.copy()
    np.fill_diagonal(temp, 0)
    row_idx, col_idx = np.unravel_index(np.argmax(temp), temp.shape)

    return {
        "best_class": best_class,
        "worst_class": worst_class,
        "most_confused_pair": f"{class_labels[row_idx]} -> {class_labels[col_idx]}",
        "most_confused_count": int(temp[row_idx, col_idx]),
        "barata_como_cara": int(cm[class_labels.index("barata"), class_labels.index("cara")]),
        "cara_como_barata": int(cm[class_labels.index("cara"), class_labels.index("barata")]),
    }


def diagnose_fit(train_f1, test_f1, train_accuracy, test_accuracy):
    f1_gap = train_f1 - test_f1
    acc_gap = train_accuracy - test_accuracy

    if (train_f1 >= 0.85 and f1_gap >= 0.10) or (train_accuracy >= 0.85 and acc_gap >= 0.10):
        return "Sobreajuste"
    if train_f1 < 0.55 and test_f1 < 0.55:
        return "Desajuste"
    if abs(f1_gap) <= 0.05 and test_f1 >= 0.60:
        return "Ajuste aceptable"
    if f1_gap >= 0.06:
        return "Ligero sobreajuste"
    return "Intermedio"


## 5. Entrenamiento y evaluación de varios modelos SVM

Se entrenan seis modelos:

- 2 con kernel lineal.
- 2 con kernel RBF.
- 2 con kernel polinomial.

Para cada uno se registra:
- tiempo de entrenamiento,
- tiempo de predicción,
- métricas en train y test,
- classification report,
- matriz de confusión,
- comentario breve sobre clases mejor y peor clasificadas.


In [ ]:
summary_rows = []
report_rows = []
confusion_notes = []

for config in model_configs:
    model_name = config["modelo"]
    print(f"Entrenando {model_name} ...")

    svm_params = {
        "kernel": config["kernel"],
        "C": config["C"],
        "decision_function_shape": "ovr",
        "cache_size": 1000,
    }
    if "gamma" in config:
        svm_params["gamma"] = config["gamma"]
    if "degree" in config:
        svm_params["degree"] = config["degree"]
    if "coef0" in config:
        svm_params["coef0"] = config["coef0"]

    pipeline = Pipeline(
        steps=[
            ("preprocess", clone(preprocessor)),
            ("model", SVC(**svm_params)),
        ]
    )

    start_train = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start_train

    start_pred = time.perf_counter()
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    pred_time = time.perf_counter() - start_pred

    metrics_train = compute_metric_block(y_train, y_pred_train)
    metrics_test = compute_metric_block(y_test, y_pred_test)

    cm_train = confusion_matrix(y_train, y_pred_train, labels=labels)
    cm_test = confusion_matrix(y_test, y_pred_test, labels=labels)

    report_train = pd.DataFrame(
        classification_report(y_train, y_pred_train, labels=labels, output_dict=True, zero_division=0)
    ).T
    report_test = pd.DataFrame(
        classification_report(y_test, y_pred_test, labels=labels, output_dict=True, zero_division=0)
    ).T

    train_summary = confusion_summary(cm_train, labels)
    test_summary = confusion_summary(cm_test, labels)

    report_train_to_save = report_train.reset_index().rename(columns={"index": "clase"})
    report_train_to_save["modelo"] = model_name
    report_train_to_save["split"] = "train"

    report_test_to_save = report_test.reset_index().rename(columns={"index": "clase"})
    report_test_to_save["modelo"] = model_name
    report_test_to_save["split"] = "test"

    report_rows.append(report_train_to_save)
    report_rows.append(report_test_to_save)

    pd.DataFrame(cm_train, index=labels, columns=labels).to_csv(
        OUTPUT_DIR / f"{model_name}_confusion_train.csv"
    )
    pd.DataFrame(cm_test, index=labels, columns=labels).to_csv(
        OUTPUT_DIR / f"{model_name}_confusion_test.csv"
    )

    hyperparams = [f"C={config['C']}"]
    if "gamma" in config:
        hyperparams.append(f"gamma={config['gamma']}")
    if "degree" in config:
        hyperparams.append(f"degree={config['degree']}")

    diagnosis = diagnose_fit(
        metrics_train["f1_macro"],
        metrics_test["f1_macro"],
        metrics_train["accuracy"],
        metrics_test["accuracy"],
    )

    summary_rows.append(
        {
            "modelo": model_name,
            "kernel": config["kernel"],
            "hiperparametros": ", ".join(hyperparams),
            "accuracy_train": metrics_train["accuracy"],
            "accuracy_test": metrics_test["accuracy"],
            "precision_macro_test": metrics_test["precision_macro"],
            "recall_macro_test": metrics_test["recall_macro"],
            "f1_train": metrics_train["f1_macro"],
            "f1_test": metrics_test["f1_macro"],
            "diferencia_f1_train_test": metrics_train["f1_macro"] - metrics_test["f1_macro"],
            "tiempo_entrenamiento_seg": train_time,
            "tiempo_prediccion_seg": pred_time,
            "diagnostico": diagnosis,
            "mejor_clase_test": test_summary["best_class"],
            "peor_clase_test": test_summary["worst_class"],
            "confusion_mas_frecuente": f"{test_summary['most_confused_pair']} ({test_summary['most_confused_count']} casos)",
            "barata_como_cara_test": test_summary["barata_como_cara"],
            "cara_como_barata_test": test_summary["cara_como_barata"],
        }
    )

    confusion_notes.append(
        {
            "modelo": model_name,
            "split": "test",
            "mejor_clase": test_summary["best_class"],
            "peor_clase": test_summary["worst_class"],
            "confusion_mas_frecuente": test_summary["most_confused_pair"],
            "casos_confusion": test_summary["most_confused_count"],
            "barata_como_cara": test_summary["barata_como_cara"],
            "cara_como_barata": test_summary["cara_como_barata"],
        }
    )

    metrics_display = pd.DataFrame(
        [
            {
                "split": "train",
                "accuracy": metrics_train["accuracy"],
                "precision_macro": metrics_train["precision_macro"],
                "recall_macro": metrics_train["recall_macro"],
                "f1_macro": metrics_train["f1_macro"],
                "precision_weighted": metrics_train["precision_weighted"],
                "recall_weighted": metrics_train["recall_weighted"],
                "f1_weighted": metrics_train["f1_weighted"],
            },
            {
                "split": "test",
                "accuracy": metrics_test["accuracy"],
                "precision_macro": metrics_test["precision_macro"],
                "recall_macro": metrics_test["recall_macro"],
                "f1_macro": metrics_test["f1_macro"],
                "precision_weighted": metrics_test["precision_weighted"],
                "recall_weighted": metrics_test["recall_weighted"],
                "f1_weighted": metrics_test["f1_weighted"],
            },
        ]
    )

    display(Markdown(f"### {model_name}"))
    display(metrics_display.round(4))
    display(report_test.round(4))
    display(
        Markdown(
            f"**Lectura rápida:** la clase mejor clasificada en test fue **{test_summary['best_class']}**; "
            f"la más difícil fue **{test_summary['worst_class']}**; la confusión más frecuente fue "
            f"**{test_summary['most_confused_pair']}** con **{test_summary['most_confused_count']}** casos."
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.heatmap(
        pd.DataFrame(cm_train, index=labels, columns=labels),
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=axes[0],
    )
    axes[0].set_title(f"{model_name} - Train")
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")

    sns.heatmap(
        pd.DataFrame(cm_test, index=labels, columns=labels),
        annot=True,
        fmt="d",
        cmap="Greens",
        ax=axes[1],
    )
    axes[1].set_title(f"{model_name} - Test")
    axes[1].set_xlabel("Predicho")
    axes[1].set_ylabel("Real")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


## 6. Tabla resumen y análisis de sobreajuste/desajuste

La siguiente tabla resume los principales indicadores para comparar todos los SVM del avance.
El criterio de diagnóstico sigue esta lógica:

- **Sobreajuste**: train alto y test claramente menor.
- **Desajuste**: train y test bajos.
- **Ajuste aceptable**: train y test cercanos con métricas razonables.


In [ ]:
results_df = pd.DataFrame(summary_rows).sort_values(
    by=["f1_test", "accuracy_test"], ascending=False
).reset_index(drop=True)

reports_df = pd.concat(report_rows, ignore_index=True)
confusion_notes_df = pd.DataFrame(confusion_notes)

results_df.to_csv(OUTPUT_DIR / "svm_summary_table.csv", index=False)
reports_df.to_csv(OUTPUT_DIR / "svm_classification_reports.csv", index=False)
confusion_notes_df.to_csv(OUTPUT_DIR / "svm_confusion_narratives.csv", index=False)

display(
    results_df[
        [
            "modelo",
            "kernel",
            "hiperparametros",
            "accuracy_train",
            "accuracy_test",
            "f1_train",
            "f1_test",
            "diferencia_f1_train_test",
            "diagnostico",
        ]
    ].round(4)
)


## 7. Texto breve listo para pegar en el avance

El siguiente bloque se genera automáticamente con base en los resultados del notebook.


In [ ]:
best_row = results_df.iloc[0]

sample_note = (
    f"No se encontró una división previa guardada en la carpeta, por lo que se creó una muestra "
    f"estratificada reproducible de {len(train_df) + len(test_df)} registros con random_state={RANDOM_STATE}. "
    "Esta misma muestra y esta misma división train/test deben mantenerse para las comparaciones posteriores."
)

summary_md = f'''
## Texto breve para el pasaporte

En este avance se cargó `listings.RData`, se limpió la variable `price` para convertirla a formato numérico y se trabajó únicamente con observaciones que tenían un precio válido. A partir del precio limpio se construyó la variable categórica `price_cat` con tres niveles: **barata**, **media** y **cara**, definidos mediante terciles del precio ({metadata['price_min']:.0f}, {metadata['price_q1']:.0f}, {metadata['price_q2']:.0f}, {metadata['price_max']:.0f}).

{sample_note}

Para preparar los datos para SVM se aplicó imputación de valores faltantes, codificación one-hot para variables categóricas y escalamiento estandarizado para variables numéricas. El escalamiento es indispensable porque SVM depende de distancias y productos punto; si una variable tiene una escala mucho mayor que las demás, altera la frontera de decisión y reduce la calidad del modelo.

Se entrenaron seis modelos SVM de clasificación: dos lineales, dos con kernel RBF y dos polinomiales. El mejor resultado de prueba se obtuvo con **{best_row['modelo']}** ({best_row['kernel']}), con **Accuracy test = {best_row['accuracy_test']:.4f}** y **F1 macro test = {best_row['f1_test']:.4f}**. La clase mejor clasificada fue **{best_row['mejor_clase_test']}**, la más difícil fue **{best_row['peor_clase_test']}** y la confusión más frecuente fue **{best_row['confusion_mas_frecuente']}**.

En términos de ajuste, el modelo líder se diagnosticó como **{best_row['diagnostico']}**, porque su desempeño en entrenamiento fue mayor que en prueba. Aun así, varios modelos mostraron un comportamiento estable, lo que permite continuar más adelante con la comparación formal frente a otros algoritmos usando exactamente la misma muestra y la misma partición guardada.
'''.strip()

display(Markdown(summary_md))
(OUTPUT_DIR / "avance_pasaporte_lab8_svm_python.md").write_text(summary_md, encoding="utf-8")

print("Archivos principales generados:")
print("-", OUTPUT_DIR / "svm_summary_table.csv")
print("-", OUTPUT_DIR / "svm_classification_reports.csv")
print("-", OUTPUT_DIR / "avance_pasaporte_lab8_svm_python.md")
